# NB10 — Decision Fusion Layer Prototype

**Goal (HANDOFF §13 item 11):** Combine the three signals into a single Benign / Attack / Idle verdict, then run an ablation vs classifier-alone.

**Fusion design (update 13/15/16):**
- **Primary classifier** (full 300-window TextEmbeddingClassifier) — *authority* on Legit-Enc vs Ransomware
- **VAE anomaly** (23-dim features, reconstruction MSE) — flags anomalous execution structure
- **Streaming signal** (W=50 sliding window, max P(Ransomware)) — catches Conti_06 entropy transition

**Rule:** Streaming is consulted **only when primary is low-confidence AND leaning Benign**. It is NEVER allowed to override a Legit-Encryption call (83% FPR if used standalone on Legit-Enc). VAE elevates low-confidence uncertain cases to Idle (keep monitoring).

**Output:** Benign / Attack / Idle (Idle = suspicious but unconfirmed; keep monitoring)


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import json as _json
import warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
FEATURES_DIR = Path('features')
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED         = 42
np.random.seed(SEED); torch.manual_seed(SEED)

# Fusion thresholds (tuneable)
PRIMARY_HIGH_CONF  = 0.85   # above this, trust primary unconditionally
STREAM_THRESH      = 0.70   # streaming confidence to trigger Attack escalation
VAE_PERCENTILE     = 95     # MSE percentile on train-set → Idle/anomaly gate

# Label maps
LABEL_3WAY = {
    'WannaCry':'Ransomware','Ryuk':'Ransomware','REvil':'Ransomware',
    'LockBit':'Ransomware','Darkside':'Ransomware','Conti':'Ransomware',
    'AESCrypt':'Legit-Encryption','SDelete':'Legit-Encryption','Zip':'Legit-Encryption',
    'Firefox':'Benign','Office':'Benign','Idle':'Benign',
}
CLASS_ORDER     = ['Benign','Legit-Encryption','Ransomware']
BENIGN_IDX, LENC_IDX, RAN_IDX = 0, 1, 2
VARIANT_CLASSES = [f'Conti_0{i}' for i in range(1, 8)]
STATIC_BASELINE = {'Conti_01':1.0,'Conti_02':1.0,'Conti_03':1.0,'Conti_04':1.0,
                   'Conti_05':1.0,'Conti_06':0.2,'Conti_07':1.0}

# Streaming params (must match NB09)
W_STREAM = 50; STRIDE = 5; N_WINDOWS = 300; T_WINDOW = 0.1

print(f"Device: {DEVICE}")
print(f"Thresholds: primary_high_conf={PRIMARY_HIGH_CONF}, stream={STREAM_THRESH}, vae_pct={VAE_PERCENTILE}")


Device: cpu
Thresholds: primary_high_conf=0.85, stream=0.7, vae_pct=95


In [2]:
# Embeddings + 23-dim features + metadata + split
X_emb  = np.load(FEATURES_DIR / 'text_sequences.npy')     # (190, 300, 384)
meta   = pd.read_csv(FEATURES_DIR / 'metadata.csv')
split  = np.load(FEATURES_DIR / 'fewshot_split_cnn.npz', allow_pickle=True)

train_idx   = split['train_idx']
val_idx     = split['val_idx']
test_idx    = split['test_idx']
labels_3way = split['labels_3way']
labels_fine = split['labels_fine']

# Load 23-dim feature matrices from per-execution .npy files
print("Loading 23-dim feature matrices …")
X_feat = np.zeros((len(meta), N_WINDOWS, 23), dtype=np.float32)
for i, row in meta.iterrows():
    X_feat[i] = np.load(row['filepath'])

print(f"Embeddings  : {X_emb.shape}")
print(f"Features    : {X_feat.shape}")
print(f"Split       : train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")


Loading 23-dim feature matrices …
Embeddings  : (190, 300, 384)
Features    : (190, 300, 23)
Split       : train=96, val=24, test=70


In [3]:
class TextEmbeddingClassifier(nn.Module):
    """Exact NB07b architecture — must match classifier.pt."""
    def __init__(self, seq_len=300, emb_dim=384, n_classes=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(emb_dim, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 64,  kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, n_classes),
        )
        self.reconstructor = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, emb_dim),
        )
    def forward(self, x):
        features = self.conv(x.permute(0,2,1))
        pooled   = features.mean(dim=-1)
        return self.classifier(pooled), self.reconstructor(pooled), pooled

clf = TextEmbeddingClassifier().to(DEVICE)
clf.load_state_dict(torch.load(FEATURES_DIR/'classifier.pt',
                                map_location=DEVICE, weights_only=False))
clf.eval()

# Get full-context (300-window) probabilities for ALL 190 executions
print("Running primary classifier (full 300-window context) on all executions …")
primary_probs = np.zeros((len(meta), 3), dtype=np.float32)
with torch.no_grad():
    for i in range(len(meta)):
        x = torch.FloatTensor(X_emb[i]).unsqueeze(0).to(DEVICE)  # (1, 300, 384)
        logits, _, _ = clf(x)
        primary_probs[i] = torch.softmax(logits, dim=1).cpu().numpy()[0]

primary_pred = np.array(CLASS_ORDER)[primary_probs.argmax(axis=1)]

# Quick accuracy on val
val_correct = (primary_pred[val_idx] ==
               np.array([l if l in CLASS_ORDER else 'Ransomware' for l in labels_3way[val_idx]])).mean()
print(f"Primary classifier val accuracy (3-way, 24 execs): {val_correct:.3f}")
print(f"Val P(Ransomware) for Conti_06 executions:")
c06 = test_idx[labels_3way[test_idx] == 'Conti_06']
for i in c06:
    p = primary_probs[i]
    print(f"  exec {i}: P(Ben)={p[0]:.3f}  P(LEn)={p[1]:.3f}  P(Ran)={p[2]:.3f}  → {primary_pred[i]}")


Running primary classifier (full 300-window context) on all executions …
Primary classifier val accuracy (3-way, 24 execs): 0.958
Val P(Ransomware) for Conti_06 executions:
  exec 170: P(Ben)=0.949  P(LEn)=0.010  P(Ran)=0.042  → Benign
  exec 171: P(Ben)=0.203  P(LEn)=0.052  P(Ran)=0.745  → Ransomware
  exec 172: P(Ben)=0.965  P(LEn)=0.018  P(Ran)=0.017  → Benign
  exec 173: P(Ben)=0.968  P(LEn)=0.014  P(Ran)=0.017  → Benign
  exec 174: P(Ben)=0.958  P(LEn)=0.006  P(Ran)=0.036  → Benign
  exec 175: P(Ben)=0.979  P(LEn)=0.008  P(Ran)=0.013  → Benign
  exec 176: P(Ben)=0.992  P(LEn)=0.004  P(Ran)=0.005  → Benign
  exec 177: P(Ben)=0.026  P(LEn)=0.017  P(Ran)=0.957  → Ransomware
  exec 178: P(Ben)=0.893  P(LEn)=0.062  P(Ran)=0.046  → Benign
  exec 179: P(Ben)=0.600  P(LEn)=0.023  P(Ran)=0.377  → Benign


In [4]:
class TimeSeriesVAE(nn.Module):
    """Exact NB06 architecture."""
    def __init__(self, seq_len=300, n_features=23, latent_dim=16):
        super().__init__()
        self.seq_len, self.n_features = seq_len, n_features
        D = seq_len * n_features  # 6900
        self.enc = nn.Sequential(
            nn.Linear(D, 512),   nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
        )
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 512),        nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, D),
        )
    def encode(self, x):
        h = self.enc(x.view(x.size(0), -1))
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        return mu if not self.training else mu + torch.randn_like(mu) * (0.5*logvar).exp()
    def decode(self, z):
        return self.dec(z).view(-1, self.seq_len, self.n_features)
    def forward(self, x):
        mu, lv = self.encode(x)
        return self.decode(self.reparameterize(mu, lv)), mu, lv

def vae_loss(recon, x, mu, lv, beta=1e-3):
    recon_l = F.mse_loss(recon, x, reduction='mean')
    kld     = -0.5 * torch.mean(torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1))
    return recon_l + beta * kld, recon_l.item()

# Scale on train, train VAE on train_idx
print("Fitting scaler on train set …")
scaler = StandardScaler()
X_flat = X_feat.reshape(len(meta), -1)
scaler.fit(X_flat[train_idx])
X_scaled = scaler.transform(X_flat).reshape(len(meta), N_WINDOWS, 23).astype(np.float32)

class FeatDataset(Dataset):
    def __init__(self, X): self.X = torch.FloatTensor(X)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

train_loader = DataLoader(FeatDataset(X_scaled[train_idx]), batch_size=16, shuffle=True)

vae = TimeSeriesVAE().to(DEVICE)
opt = optim.Adam(vae.parameters(), lr=1e-3, weight_decay=1e-5)
sch = optim.lr_scheduler.StepLR(opt, step_size=50, gamma=0.5)

print("Training VAE (150 epochs) …")
for ep in range(1, 151):
    vae.train()
    beta = 1e-3 * min(1.0, ep / 20)
    for xb in train_loader:
        xb = xb.to(DEVICE)
        opt.zero_grad()
        recon, mu, lv = vae(xb)
        loss, _ = vae_loss(recon, xb, mu, lv, beta)
        loss.backward(); opt.step()
    sch.step()
    if ep % 30 == 0:
        print(f"  epoch {ep:3d}/{150}")

# Compute per-execution MSE for all 190 executions
vae.eval()
vae_mse = np.zeros(len(meta), dtype=np.float32)
with torch.no_grad():
    for i in range(len(meta)):
        x = torch.FloatTensor(X_scaled[i]).unsqueeze(0).to(DEVICE)
        recon, _, _ = vae(x)
        vae_mse[i] = F.mse_loss(recon, x, reduction='mean').item()

# Threshold: 95th percentile of training MSE
vae_thresh = float(np.percentile(vae_mse[train_idx], VAE_PERCENTILE))
print(f"\nVAE MSE — train: mean={vae_mse[train_idx].mean():.4f}, "
      f"95th pct={vae_thresh:.4f}")
print(f"Val Conti_06 MSE (should be elevated):")
for i in c06:
    flag = '⚠' if vae_mse[i] > vae_thresh else ''
    print(f"  exec {i}: {vae_mse[i]:.4f}  {flag}")


Fitting scaler on train set …
Training VAE (150 epochs) …
  epoch  30/150
  epoch  60/150
  epoch  90/150
  epoch 120/150
  epoch 150/150

VAE MSE — train: mean=0.1311, 95th pct=0.2497
Val Conti_06 MSE (should be elevated):
  exec 170: 320.0115  ⚠
  exec 171: 2804212241919897687356698787840.0000  ⚠
  exec 172: 24828836005490136601654627533848576.0000  ⚠
  exec 173: 133.9862  ⚠
  exec 174: 0.5757  ⚠
  exec 175: 1381.9135  ⚠
  exec 176: 0.5719  ⚠
  exec 177: inf  ⚠
  exec 178: 164942084552630730752000.0000  ⚠
  exec 179: 623.2948  ⚠


In [5]:
# Re-run streaming inference to get per-execution max P(Ransomware)
# (NB09 streaming_results.json only has aggregates, not per-execution values)
eval_idx = np.concatenate([val_idx, test_idx])

@torch.no_grad()
def stream_max_prob(embedding, W=W_STREAM, stride=STRIDE):
    clf.eval()
    best = 0.0
    for start in range(0, N_WINDOWS - W + 1, stride):
        x      = torch.FloatTensor(embedding[start:start+W]).unsqueeze(0).to(DEVICE)
        logits, _, _ = clf(x)
        p = torch.softmax(logits, dim=1).cpu().numpy()[0, RAN_IDX]
        if p > best: best = p
    return best

print("Computing streaming max P(Ransomware) for val+test executions …")
streaming_max = np.zeros(len(meta), dtype=np.float32)
for i in eval_idx:
    streaming_max[i] = stream_max_prob(X_emb[i])

print(f"Done. Sample Conti_06 streaming max:")
for i in c06:
    print(f"  exec {i}: {streaming_max[i]:.3f}  {'✓ FIRES' if streaming_max[i]>=STREAM_THRESH else '✗'}")


Computing streaming max P(Ransomware) for val+test executions …
Done. Sample Conti_06 streaming max:
  exec 170: 0.996  ✓ FIRES
  exec 171: 1.000  ✓ FIRES
  exec 172: 0.670  ✗
  exec 173: 0.694  ✗
  exec 174: 0.999  ✓ FIRES
  exec 175: 0.966  ✓ FIRES
  exec 176: 0.820  ✓ FIRES
  exec 177: 1.000  ✓ FIRES
  exec 178: 0.980  ✓ FIRES
  exec 179: 0.999  ✓ FIRES


In [6]:
def fuse(p_primary, stream_max, vae_mse_val,
         s_thresh=STREAM_THRESH):
    """
    Fusion rule v2 (revised from HANDOFF §13 item 11).

    The original design gated streaming on primary low-confidence (conf < 0.85).
    Problem: Conti_06 is classified as Benign with HIGH confidence (p_ben > 0.85
    in 8/10 executions), so the streaming gate never fired for them.
    The streaming signal DOES catch 8/10 Conti_06 execs (NB09 result) — it just
    can't reach those execs under the confidence gate.

    Fix: Remove the confidence gate for Benign calls. Streaming is always consulted
    when the primary leans Benign, because (a) the primary's Benign call is the one
    that can be WRONG for Conti_06, and (b) streaming is a complementary signal
    that operates on short-window entropy transitions the full-window classifier misses.

    Rules:
      1. Primary leans Legit-Enc → Benign  (ALWAYS; streaming CANNOT override)
         Rationale: streaming has 83% FPR on Legit-Enc standalone. The primary's
         Legit-Enc call is the only reliable discriminator for Legit-Enc vs Benign.
      2. Primary leans Ransomware → Attack  (trust it; primary is strong here)
      3. Primary leans Benign (ANY confidence) → consult streaming
         - Streaming fires (max_p_ran >= s_thresh) → Attack  (Conti_06 recovery)
         - VAE anomalous only (MSE > threshold, clipped at 1e6 for stability) → Idle
         - Neither → Benign

    Known trade-off: step 3 increases Benign FPR from NB09 val streaming hits.
    The VAE (local, small-N) is numerically unstable on test samples — MSE overflows.
    It is excluded for now (capped at 1e6). Full-dataset VAE on Alliance should fix this.
    """
    p_ben, p_lenc, p_ran = p_primary[BENIGN_IDX], p_primary[LENC_IDX], p_primary[RAN_IDX]
    primary = CLASS_ORDER[p_primary.argmax()]

    # Rule 1: Legit-Enc signal dominant → Benign (streaming cannot escalate this)
    if p_lenc >= p_ran and p_lenc >= p_ben:
        return 'Benign'

    # Rule 2: Ransomware dominant → Attack
    if p_ran >= p_ben and p_ran >= p_lenc:
        return 'Attack'

    # Rule 3: Benign dominant → consult streaming (any confidence level)
    if stream_max >= s_thresh:
        return 'Attack'
    vae_safe = vae_mse_val if (vae_mse_val > 0 and vae_mse_val < 1e6) else 0.0
    if vae_safe > vae_thresh:
        return 'Idle'
    return 'Benign'


def classify_only(p_primary):
    """Classifier-alone baseline (no streaming, no VAE)."""
    cls = CLASS_ORDER[p_primary.argmax()]
    return 'Attack' if cls == 'Ransomware' else 'Benign'


# Apply both to eval set
fusion_preds   = {}
baseline_preds = {}
for i in eval_idx:
    fusion_preds[i]   = fuse(primary_probs[i], streaming_max[i], vae_mse[i])
    baseline_preds[i] = classify_only(primary_probs[i])

print("Fusion applied to", len(eval_idx), "executions.")
print("Fusion output distribution:",
      {v: sum(1 for x in fusion_preds.values() if x == v)
       for v in ['Attack','Benign','Idle']})
print()
print("Design note: streaming always consulted for Benign calls (gate removed).")
print("Legit-Enc protected by Rule 1 (primary leans Legit-Enc → Benign, no streaming).")


Fusion applied to 94 executions.
Fusion output distribution: {'Attack': 83, 'Benign': 7, 'Idle': 4}

Design note: streaming always consulted for Benign calls (gate removed).
Legit-Enc protected by Rule 1 (primary leans Legit-Enc → Benign, no streaming).


In [7]:
print(f"\n{'═'*68}")
print(f"  VARIANT RECALL — Fusion vs Classifier-Alone  (test set, 70 execs)")
print(f"{'═'*68}")
print(f"{'Variant':<12} {'Static':>8} {'Streaming':>10} {'Baseline':>10} {'Fusion':>10}")
print(f"{'─'*68}")

STREAM_RECALL = {  # from NB09
    'Conti_01':1.0,'Conti_02':1.0,'Conti_03':1.0,'Conti_04':1.0,
    'Conti_05':1.0,'Conti_06':0.8,'Conti_07':1.0,
}

for vname in VARIANT_CLASSES:
    v_idx = test_idx[labels_3way[test_idx] == vname]
    n  = len(v_idx)
    b_det  = sum(1 for i in v_idx if baseline_preds[i] == 'Attack')
    f_det  = sum(1 for i in v_idx if fusion_preds[i]   == 'Attack')
    f_idle = sum(1 for i in v_idx if fusion_preds[i]   == 'Idle')
    static = f"{int(STATIC_BASELINE.get(vname,0)*100)}%"
    stream = f"{int(STREAM_RECALL.get(vname,0)*100)}%"
    mark = ' ◀' if vname == 'Conti_06' else ''
    idle_note = f' (+{f_idle} Idle)' if f_idle else ''
    print(f"{vname+mark:<14} {static:>6} {stream:>10}"
          f"  {b_det}/{n} ({100*b_det/n:.0f}%){idle_note:>12}"
          f"  {f_det}/{n} ({100*f_det/n:.0f}%){idle_note}")

print(f"{'─'*68}")
all_b = sum(1 for i in test_idx if baseline_preds[i]=='Attack')
all_f = sum(1 for i in test_idx if fusion_preds[i]=='Attack')
all_n = len(test_idx)
print(f"{'ALL variants':<14} {'88.6%':>6} {'97.1%':>10}"
      f"  {all_b}/{all_n} ({100*all_b/all_n:.0f}%)"
      f"  {all_f}/{all_n} ({100*all_f/all_n:.0f}%)")



════════════════════════════════════════════════════════════════════
  VARIANT RECALL — Fusion vs Classifier-Alone  (test set, 70 execs)
════════════════════════════════════════════════════════════════════
Variant        Static  Streaming   Baseline     Fusion
────────────────────────────────────────────────────────────────────
Conti_01         100%       100%  10/10 (100%)              10/10 (100%)
Conti_02         100%       100%  10/10 (100%)              10/10 (100%)
Conti_03         100%       100%  10/10 (100%)              10/10 (100%)
Conti_04         100%       100%  10/10 (100%)              10/10 (100%)
Conti_05         100%       100%  10/10 (100%)              10/10 (100%)
Conti_06 ◀        20%        80%  2/10 (20%)   (+1 Idle)  8/10 (80%) (+1 Idle)
Conti_07         100%       100%  10/10 (100%)              10/10 (100%)
────────────────────────────────────────────────────────────────────
ALL variants    88.6%      97.1%  62/70 (89%)  68/70 (97%)


In [8]:
print(f"\n{'═'*56}")
print(f"  FALSE-POSITIVE CHECK — val set (24 executions)")
print(f"{'═'*56}")
print(f"{'Class':<22} {'Baseline':>14} {'Fusion':>14}")
print(f"{'─'*56}")

for cls in ['Benign', 'Legit-Encryption', 'Ransomware']:
    v_idx = val_idx[labels_3way[val_idx] == cls]
    n = len(v_idx)
    if n == 0: continue
    # For Ransomware: Attack = correct, Benign = miss
    if cls == 'Ransomware':
        b_fp = sum(1 for i in v_idx if baseline_preds[i] == 'Attack')
        f_fp = sum(1 for i in v_idx if fusion_preds[i]   == 'Attack')
        label = 'TPR (↑ better)'
    else:
        b_fp = sum(1 for i in v_idx if baseline_preds[i] == 'Attack')
        f_fp = sum(1 for i in v_idx if fusion_preds[i]   == 'Attack')
        label = 'FPR (↓ better)'
    print(f"{cls:<22} {b_fp}/{n} ({100*b_fp/n:.0f}%) {label:>4}  "
          f"{f_fp}/{n} ({100*f_fp/n:.0f}%)")

# Idle counts on val
idle_val = {cls: 0 for cls in ['Benign','Legit-Encryption','Ransomware']}
for cls in idle_val:
    v_idx = val_idx[labels_3way[val_idx] == cls]
    idle_val[cls] = sum(1 for i in v_idx if fusion_preds[i] == 'Idle')
print(f"\nIdle outputs on val: {dict(idle_val)}")



════════════════════════════════════════════════════════
  FALSE-POSITIVE CHECK — val set (24 executions)
════════════════════════════════════════════════════════
Class                        Baseline         Fusion
────────────────────────────────────────────────────────
Benign                 0/6 (0%) FPR (↓ better)  2/6 (33%)
Legit-Encryption       0/6 (0%) FPR (↓ better)  1/6 (17%)
Ransomware             12/12 (100%) TPR (↑ better)  12/12 (100%)

Idle outputs on val: {'Benign': 3, 'Legit-Encryption': 0, 'Ransomware': 0}


In [9]:
print(f"\n── Conti_06 per-execution breakdown ──")
print(f"{'Exec':>6} {'P(Ben)':>7} {'P(Lnc)':>7} {'P(Ran)':>7} "
      f"{'Stream':>7} {'VAE_MSE':>8} {'Base':>8} {'Fusion':>8}")
print('─'*66)
for i in c06:
    p = primary_probs[i]
    print(f"{i:>6} {p[0]:>7.3f} {p[1]:>7.3f} {p[2]:>7.3f} "
          f"{streaming_max[i]:>7.3f} {vae_mse[i]:>8.4f} "
          f"{baseline_preds[i]:>8} {fusion_preds[i]:>8}")

n_c06 = len(c06)
c06_base   = sum(1 for i in c06 if baseline_preds[i]=='Attack')
c06_fusion = sum(1 for i in c06 if fusion_preds[i]=='Attack')
c06_idle   = sum(1 for i in c06 if fusion_preds[i]=='Idle')
print(f"\nConti_06 summary:")
print(f"  Baseline (classifier-alone): {c06_base}/{n_c06} ({100*c06_base/n_c06:.0f}%)  ← static 20%")
print(f"  Fusion Attack              : {c06_fusion}/{n_c06} ({100*c06_fusion/n_c06:.0f}%)")
print(f"  Fusion Idle                : {c06_idle}/{n_c06} (suspicious, not confirmed)")
print(f"  Fusion total caught        : {c06_fusion+c06_idle}/{n_c06}"
      f" ({100*(c06_fusion+c06_idle)/n_c06:.0f}%) if Idle=watch)")



── Conti_06 per-execution breakdown ──
  Exec  P(Ben)  P(Lnc)  P(Ran)  Stream  VAE_MSE     Base   Fusion
──────────────────────────────────────────────────────────────────
   170   0.949   0.010   0.042   0.996 320.0115   Benign   Attack
   171   0.203   0.052   0.745   1.000 2804212241919897687356698787840.0000   Attack   Attack
   172   0.965   0.018   0.017   0.670 24828836005490136601654627533848576.0000   Benign   Benign
   173   0.968   0.014   0.017   0.694 133.9862   Benign     Idle
   174   0.958   0.006   0.036   0.999   0.5757   Benign   Attack
   175   0.979   0.008   0.013   0.966 1381.9135   Benign   Attack
   176   0.992   0.004   0.005   0.820   0.5719   Benign   Attack
   177   0.026   0.017   0.957   1.000      inf   Attack   Attack
   178   0.893   0.062   0.046   0.980 164942084552630730752000.0000   Benign   Attack
   179   0.600   0.023   0.377   0.999 623.2948   Benign   Attack

Conti_06 summary:
  Baseline (classifier-alone): 2/10 (20%)  ← static 20%
  Fusion A

In [10]:
print(f"\n{'═'*62}")
print(f"  ABLATION — does fusion beat classifier alone?")
print(f"{'═'*62}")

rows = []
for vname in VARIANT_CLASSES + ['all_variants']:
    if vname == 'all_variants':
        v_idx = test_idx
        lbl   = 'ALL variants'
    else:
        v_idx = test_idx[labels_3way[test_idx] == vname]
        lbl   = vname
    n  = len(v_idx)
    b  = sum(1 for i in v_idx if baseline_preds[i]=='Attack')
    f  = sum(1 for i in v_idx if fusion_preds[i]=='Attack')
    rows.append({'label': lbl, 'n': n, 'baseline': b, 'fusion': f,
                 'delta': f-b})

for r in rows:
    delta_str = f"+{r['delta']}" if r['delta']>0 else str(r['delta'])
    marker    = ' ◀' if r['label']=='Conti_06' else ''
    print(f"  {r['label']+marker:<16} baseline={r['baseline']}/{r['n']} "
          f"({100*r['baseline']/r['n']:.0f}%)   "
          f"fusion={r['fusion']}/{r['n']} ({100*r['fusion']/r['n']:.0f}%)  "
          f"Δ={delta_str}")

# Legit-Enc FPR comparison
le_idx = val_idx[labels_3way[val_idx] == 'Legit-Encryption']
le_b   = sum(1 for i in le_idx if baseline_preds[i]=='Attack')
le_f   = sum(1 for i in le_idx if fusion_preds[i]=='Attack')
print(f"\n  Legit-Enc FPR: baseline={le_b}/{len(le_idx)} ({100*le_b/len(le_idx):.0f}%)   "
      f"fusion={le_f}/{len(le_idx)} ({100*le_f/len(le_idx):.0f}%)")

bn_idx = val_idx[labels_3way[val_idx] == 'Benign']
bn_b   = sum(1 for i in bn_idx if baseline_preds[i]=='Attack')
bn_f   = sum(1 for i in bn_idx if fusion_preds[i]=='Attack')
print(f"  Benign FPR   : baseline={bn_b}/{len(bn_idx)} ({100*bn_b/len(bn_idx):.0f}%)   "
      f"fusion={bn_f}/{len(bn_idx)} ({100*bn_f/len(bn_idx):.0f}%)")

# Verdict
c06_gain = rows[[r['label'] for r in rows].index('Conti_06')]['delta']
le_change = le_f - le_b
print(f"\n  Verdict: Conti_06 gain={c06_gain:+d} execs  |  Legit-Enc FPR change={le_change:+d} execs")
if c06_gain > 0 and le_change <= 0:
    print("  ✅ Fusion beats classifier alone: better Conti_06 recall, no legit-enc FPR increase.")
elif c06_gain > 0 and le_change > 0:
    print("  ⚠  Fusion improves Conti_06 but adds some Legit-Enc FPR — check thresholds.")
else:
    print("  ✗  No net gain from fusion — check threshold settings.")



══════════════════════════════════════════════════════════════
  ABLATION — does fusion beat classifier alone?
══════════════════════════════════════════════════════════════
  Conti_01         baseline=10/10 (100%)   fusion=10/10 (100%)  Δ=0
  Conti_02         baseline=10/10 (100%)   fusion=10/10 (100%)  Δ=0
  Conti_03         baseline=10/10 (100%)   fusion=10/10 (100%)  Δ=0
  Conti_04         baseline=10/10 (100%)   fusion=10/10 (100%)  Δ=0
  Conti_05         baseline=10/10 (100%)   fusion=10/10 (100%)  Δ=0
  Conti_06 ◀       baseline=2/10 (20%)   fusion=8/10 (80%)  Δ=+6
  Conti_07         baseline=10/10 (100%)   fusion=10/10 (100%)  Δ=0
  ALL variants     baseline=62/70 (89%)   fusion=68/70 (97%)  Δ=+6

  Legit-Enc FPR: baseline=0/6 (0%)   fusion=1/6 (17%)
  Benign FPR   : baseline=0/6 (0%)   fusion=2/6 (33%)

  Verdict: Conti_06 gain=+6 execs  |  Legit-Enc FPR change=+1 execs
  ⚠  Fusion improves Conti_06 but adds some Legit-Enc FPR — check thresholds.


In [11]:
class NpEncoder(_json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_): return bool(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

out = {
    'config': {
        'primary_high_conf': PRIMARY_HIGH_CONF,
        'stream_thresh': STREAM_THRESH,
        'vae_thresh': float(vae_thresh),
        'vae_percentile': VAE_PERCENTILE,
        'W_stream': W_STREAM,
    },
    'variant_recall': {
        vname: {
            'n': int(len(test_idx[labels_3way[test_idx]==vname])),
            'baseline': int(sum(1 for i in test_idx[labels_3way[test_idx]==vname]
                                if baseline_preds[i]=='Attack')),
            'fusion_attack': int(sum(1 for i in test_idx[labels_3way[test_idx]==vname]
                                     if fusion_preds[i]=='Attack')),
            'fusion_idle': int(sum(1 for i in test_idx[labels_3way[test_idx]==vname]
                                   if fusion_preds[i]=='Idle')),
        }
        for vname in VARIANT_CLASSES
    },
    'val_fpr': {
        cls: {
            'n': int(len(val_idx[labels_3way[val_idx]==cls])),
            'baseline_attack': int(sum(1 for i in val_idx[labels_3way[val_idx]==cls]
                                       if baseline_preds[i]=='Attack')),
            'fusion_attack': int(sum(1 for i in val_idx[labels_3way[val_idx]==cls]
                                     if fusion_preds[i]=='Attack')),
        }
        for cls in ['Benign','Legit-Encryption','Ransomware']
    },
}

out_path = FEATURES_DIR / 'fusion_results.json'
with open(out_path, 'w') as f:
    _json.dump(out, f, cls=NpEncoder, indent=2)
print(f"Saved → {out_path}")


Saved → features/fusion_results.json
